# 03 - Embeat Recommendation Model Training

## Mục tiêu
Notebook này trình bày quy trình huấn luyện và kiểm tra model gợi ý âm nhạc Embeat:

1. Nạp dữ liệu đã tiền xử lý.
2. Chuẩn hóa 9 đặc trưng âm học bằng `StandardScaler`.
3. Mã hóa thể loại bằng `MultiLabelBinarizer` và Sparse Matrix.
4. Huấn luyện model và lưu artifact.
5. Tải lại artifact, kiểm tra cấu trúc model và thử recommendation.
6. Kiểm tra incremental ingestion theo batch bằng `partial_fit()` và Parquet tạm.

**Phạm vi thực nghiệm:** 29.985 tracks sau làm sạch, 9 audio features và 4.211 genres nếu sử dụng dataset `balanced_eda.csv` hiện tại. Đây không phải mô hình được huấn luyện trên toàn bộ Embeat 45M.

> Ghi chú: `fit_incremental()` cập nhật `StandardScaler` theo từng batch bằng `partial_fit()` và ghi dữ liệu batch ra Parquet tạm, không giữ danh sách DataFrame trong `_pending_batches`. Sau khi nhận đủ batch, `finalize_incremental_training()` tạo DataFrame và các ma trận cuối cùng trong RAM theo contract hiện tại của recommender. Đây là incremental batch ingestion/scaler fitting, chưa phải hệ thống recommendation hoàn toàn online hoặc out-of-core.

Notebook được tinh gọn để mỗi cell chỉ hiển thị kết quả cần thiết; các cell audit, debug, in source code và Git diff được đưa ra khỏi luồng huấn luyện chính.


In [97]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Khởi tạo môi trường và đường dẫn

In [98]:
from pathlib import Path
import sys
import json
import time
import pandas as pd

# Hỗ trợ Google Colab và chạy local trong thư mục project.
COLAB_ROOT = Path("/content/drive/MyDrive/music_recommender_project")
LOCAL_ROOT = Path.cwd()

if (COLAB_ROOT / "src").exists():
    PROJECT_ROOT = COLAB_ROOT
elif (LOCAL_ROOT / "src").exists():
    PROJECT_ROOT = LOCAL_ROOT
else:
    raise FileNotFoundError(
        "Không tìm thấy thư mục project. Hãy mở notebook từ thư mục music_recommender_project."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import recommender trước, sau đó đồng bộ class reference trong src.train.
# Điều này tránh PicklingError nếu notebook từng reload src.recommender.
import src.recommender as recommender_module
import src.train as train_module

EmbeatRecommender = recommender_module.EmbeatRecommender
train_module.EmbeatRecommender = EmbeatRecommender
train_model = train_module.train_model

assert train_module.EmbeatRecommender is EmbeatRecommender, (
    "Class reference giữa src.train và src.recommender chưa đồng bộ."
)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "balanced_eda.csv"
ARTIFACT_PATH = PROJECT_ROOT / "artifacts" / "embeat_recommender.joblib"
METADATA_PATH = ARTIFACT_PATH.with_name(
    f"{ARTIFACT_PATH.stem}_metadata.json"
)

assert DATA_PATH.exists(), f"Không tìm thấy dữ liệu: {DATA_PATH}"

# Được đặt lại ở cell training; dùng để chặn việc đọc artifact/metadata cũ.
TRAINING_STARTED_AT = None
model = None

print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_PATH.name)
print("Artifact output:", ARTIFACT_PATH)
print("Class identity: SYNCED")
print("ENVIRONMENT: READY")


Project: /content/drive/MyDrive/music_recommender_project
Dataset: balanced_eda.csv
Artifact output: /content/drive/MyDrive/music_recommender_project/artifacts/embeat_recommender.joblib
Class identity: SYNCED
ENVIRONMENT: READY


## 2. Nạp và kiểm tra dữ liệu

In [99]:
df = pd.read_csv(DATA_PATH)

print("Số dòng:", f"{len(df):,}")
print("Số cột:", len(df.columns))
print("Track ID duy nhất:", f"{df['track_id'].nunique():,}")
print("Dữ liệu trùng Track ID:", int(df['track_id'].duplicated().sum()))

required_columns = [
    "track_id", "track_name", "artist_name", "artist_genres",
    "popularity", "danceability", "energy", "valence", "tempo",
    "acousticness", "instrumentalness", "speechiness", "liveness", "loudness"
]
missing = [col for col in required_columns if col not in df.columns]
assert not missing, f"Thiếu cột bắt buộc: {missing}"

display(df.head(5))
print("DATA VALIDATION: PASS")

Số dòng: 30,000
Số cột: 28
Track ID duy nhất: 30,000
Dữ liệu trùng Track ID: 0


,track_id,track_name,isrc,popularity,explicit,artist_idx,artist_id,artist_name,artist_popularity,artist_genres,...,mode,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,source_shard
0,7fEoXCZTZFosUFvFQg1BmW,Took Her To The O,USUYG1296663,0.82,1,380,6QtgPSJPSzcnn7dPZ4VINp,King Von,0.79,chicago rap,...,1,0.820,0.592,-7.002,0.2900,0.0149,0.0000,0.1210,0.400,1
1,7fW5TnCdNFhdPw7C2QakEO,Nossas Fotos,BCOOP2500007,0.82,0,819,7cmuxPnXRJxwuYDHfbD8Eu,Zé Felipe,0.75,"arrocha, agronejo, sertanejo universitario, se...",...,1,0.849,0.772,-3.856,0.1230,0.2250,0.0000,0.2490,0.866,1
2,7iL6o9tox1zgHpKUfh9vuC,In Da Club,USIR10300033,0.82,1,123,3q7HBObVc0L8jNeTe5Gofh,50 Cent,0.84,"hip hop, gangster rap, east coast hip hop, rap...",...,0,0.899,0.713,-2.752,0.3660,0.2550,0.0000,0.0708,0.777,1
3,7j74lucZ59vqN67Ipe2ZcY,Maneater,USRC10301821,0.82,0,814,77tT1kLj6mCWtFNqiOmP9H,Daryl Hall & John Oates,0.75,"mellow gold, rock, heartland rock, classic roc...",...,0,0.724,0.498,-13.083,0.0423,0.0405,0.0001,0.0761,0.823,1
4,7m18pBAwjGPROTAf6V2cbw,Don’t Say You Love Me,USA2P2509874,0.82,0,458,5vV3bFXnN6D6N3Nj4xRvaV,Jin,0.78,k-pop,...,1,0.729,0.562,-5.490,0.0304,0.4840,0.0000,0.1050,0.757,1


DATA VALIDATION: PASS


## 3. Huấn luyện model và lưu artifact

In [100]:
# Ghi mốc thời gian để phân biệt artifact mới với file còn sót từ lần chạy trước.
TRAINING_STARTED_AT = time.time()
model = None

model = train_model(
    input_path=DATA_PATH,
    output_path=ARTIFACT_PATH
)

assert ARTIFACT_PATH.exists(), "Model artifact chưa được tạo."
assert ARTIFACT_PATH.stat().st_mtime >= TRAINING_STARTED_AT - 1, (
    "Artifact không được tạo/cập nhật trong lần chạy training hiện tại."
)

print("TRAINING: COMPLETED")
print("Tracks sau làm sạch:", f"{len(model.df):,}")
print("Audio matrix:", model.feature_matrix.shape)
print("Genre matrix:", model.genre_matrix.shape)
print("Artifact size (MB):", round(ARTIFACT_PATH.stat().st_size / (1024 ** 2), 2))


EMBEAT MODEL TRAINING

[1/5] Loading dataset...
Original shape: (30000, 28)

[2/5] Training audio model...
Cleaned tracks: 29985
Audio matrix: (29985, 9)

[3/5] Building genre matrix...
Genre matrix: (29985, 4211)
Unique genres: 4211

[4/5] Saving model artifact...
Model saved successfully: /content/drive/MyDrive/music_recommender_project/artifacts/embeat_recommender.joblib

[5/5] Generating training metadata...
Metadata saved: /content/drive/MyDrive/music_recommender_project/artifacts/embeat_recommender_metadata.json
TRAINING: COMPLETED
Tracks sau làm sạch: 29,985
Audio matrix: (29985, 9)
Genre matrix: (29985, 4211)
Artifact size (MB): 6.62


## 4. Kiểm tra metadata huấn luyện

In [101]:
assert model is not None, "Cell training chưa chạy thành công trong runtime hiện tại."
assert TRAINING_STARTED_AT is not None, "Thiếu mốc thời gian training hiện tại."
assert METADATA_PATH.exists(), f"Không tìm thấy metadata: {METADATA_PATH}"
assert METADATA_PATH.stat().st_mtime >= TRAINING_STARTED_AT - 1, (
    "Metadata là file cũ từ lần training trước, không thuộc lần chạy hiện tại."
)

with open(METADATA_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

print(json.dumps(metadata, indent=2, ensure_ascii=False))

assert metadata.get("training_method") == "csv", (
    f"Metadata không thuộc CSV training: {metadata.get('training_method')}"
)
assert metadata.get("artifact_path") == ARTIFACT_PATH.name
assert metadata.get("artifact_filename") == ARTIFACT_PATH.name
assert metadata.get("total_tracks") == len(model.df)
assert metadata.get("audio_matrix_shape") == list(model.feature_matrix.shape)
assert metadata.get("genre_matrix_shape") == list(model.genre_matrix.shape)

# Không cho phép metadata lưu đường dẫn tuyệt đối của máy/Colab.
for key, value in metadata.items():
    if isinstance(value, str):
        assert not value.startswith(("/content/", "/mnt/", "C:\\")), (
            f"Metadata chứa đường dẫn tuyệt đối tại {key}: {value}"
        )

print("METADATA FILE:", METADATA_PATH.name)
print("METADATA CURRENT-RUN CHECK: PASS")
print("METADATA PORTABILITY: PASS")


{
  "model_name": "EmbeatRecommender",
  "training_method": "csv",
  "total_tracks": 29985,
  "audio_features": 9,
  "total_genres": 4211,
  "audio_matrix_shape": [
    29985,
    9
  ],
  "genre_matrix_shape": [
    29985,
    4211
  ],
  "artifact_path": "embeat_recommender.joblib",
  "artifact_filename": "embeat_recommender.joblib"
}
METADATA FILE: embeat_recommender_metadata.json
METADATA CURRENT-RUN CHECK: PASS
METADATA PORTABILITY: PASS


## 5. Tải lại artifact và kiểm tra tính toàn vẹn

In [102]:
assert TRAINING_STARTED_AT is not None
assert ARTIFACT_PATH.exists()
assert ARTIFACT_PATH.stat().st_mtime >= TRAINING_STARTED_AT - 1, (
    "Artifact không thuộc lần training hiện tại."
)

loaded_model = EmbeatRecommender.load_artifact(str(ARTIFACT_PATH))

assert len(loaded_model.df) == len(model.df)
assert loaded_model.feature_matrix.shape == model.feature_matrix.shape
assert loaded_model.genre_matrix.shape == model.genre_matrix.shape
assert loaded_model.df["track_id"].tolist() == model.df["track_id"].tolist()

print("Loaded tracks:", f"{len(loaded_model.df):,}")
print("Audio matrix:", loaded_model.feature_matrix.shape)
print("Genre matrix:", loaded_model.genre_matrix.shape)
print("TRACK ORDER: PASS")
print("CURRENT ARTIFACT RELOAD: PASS")


Loaded tracks: 29,985
Audio matrix: (29985, 9)
Genre matrix: (29985, 4211)
TRACK ORDER: PASS
CURRENT ARTIFACT RELOAD: PASS


## 6. Kiểm tra thử chức năng gợi ý

In [103]:
query_track_id = loaded_model.df.iloc[0]["track_id"]

recommendations = loaded_model.recommend_hybrid_mmr(
    track_id=query_track_id,
    top_k=5,
    as_dict=False
)

assert len(recommendations) == 5
assert query_track_id not in recommendations["track_id"].values

display(
    recommendations[
        ["track_name", "artist_name", "artist_genres",
         "audio_similarity", "genre_similarity",
         "hybrid_similarity", "mmr_score"]
    ]
)

print("RECOMMENDATION SMOKE TEST: PASS")

,track_name,artist_name,artist_genres,audio_similarity,genre_similarity,hybrid_similarity,mmr_score
0,Pop Out (feat. Lil Tjay),Polo G,"rap, chicago rap",0.955527,0.853553,0.924935,0.594909
1,1400 / 999 Freestyle (feat. Juice WRLD),Juice WRLD,"rap, melodic rap, chicago rap",0.919594,0.788675,0.880318,0.339705
2,Flex (feat. Juice WRLD),Juice WRLD,"rap, melodic rap, chicago rap",0.965236,0.788675,0.912268,0.315116
3,Back Again (feat. Lil Durk),Lil Durk,"chicago drill, rap, chicago rap, trap, drill, ...",0.930871,0.704124,0.862847,0.295876
4,Wasted (feat. Lil Uzi Vert),Juice WRLD,"rap, melodic rap, chicago rap",0.963020,0.788675,0.910716,0.290530


RECOMMENDATION SMOKE TEST: PASS


## 7. Kiểm tra Incremental Training (tùy chọn)

Cell này tạo model thử nghiệm riêng, nhận ba batch bằng `fit_incremental()`, xác nhận không tích lũy `_pending_batches`, kiểm tra batch được ghi ra Parquet tạm và finalize thành công.

Lưu ý: `StandardScaler` được cập nhật theo batch bằng `partial_fit()`. Sau finalize, DataFrame và ma trận recommender cuối cùng vẫn được dựng trong RAM.


In [104]:
import numpy as np

batch_test_model = EmbeatRecommender()
row_splits = np.array_split(np.arange(len(df)), 3)
batches = [df.iloc[indices].copy() for indices in row_splits if len(indices)]

for batch_index, batch in enumerate(batches):
    batch_test_model.fit_incremental(
        batch,
        is_first_batch=(batch_index == 0)
    )

assert not hasattr(batch_test_model, "_pending_batches"), (
    "Recommender vẫn còn cơ chế lưu DataFrame batches trong RAM."
)
assert batch_test_model._incremental_batch_count == len(batches)

received_batches = batch_test_model._incremental_batch_count
received_rows = batch_test_model._incremental_track_count
temp_parquet_path = batch_test_model._incremental_path

assert received_batches == 3
assert received_rows > 0
assert temp_parquet_path is not None and temp_parquet_path.exists()
assert batch_test_model._incremental_scaler is not None

# StandardScaler.partial_fit đã nhìn thấy toàn bộ số dòng đã clean/deduplicate.
scaler_rows = int(np.asarray(
    batch_test_model._incremental_scaler.n_samples_seen_
).max())
assert scaler_rows == received_rows, (
    f"Scaler đã thấy {scaler_rows} rows nhưng incremental pipeline ghi nhận {received_rows}."
)

batch_test_model.finalize_incremental_training()

assert batch_test_model.feature_matrix.shape == (len(batch_test_model.df), 9)
assert batch_test_model.genre_matrix.shape[0] == len(batch_test_model.df)
assert len(batch_test_model.id_to_idx) == len(batch_test_model.df)
assert batch_test_model.scaler is not None
assert not hasattr(batch_test_model, "_pending_batches")

print("Received batches:", received_batches)
print("Received unique/cleaned rows:", f"{received_rows:,}")
print("Scaler samples seen:", scaler_rows)
print("Cleaned rows after finalize:", f"{len(batch_test_model.df):,}")
print("Audio matrix:", batch_test_model.feature_matrix.shape)
print("Genre matrix:", batch_test_model.genre_matrix.shape)
print("PARQUET + PARTIAL_FIT INCREMENTAL TEST: PASS")


=== FINALIZING INCREMENTAL TRAINING ===
Received batches: 3
Total unique tracks: 29985
Cleaned tracks: 29985
Audio matrix: (29985, 9)
Genre matrix: (29985, 4211)
INCREMENTAL TRAINING COMPLETED
Received batches: 3
Received unique/cleaned rows: 29,985
Scaler samples seen: 29985
Cleaned rows after finalize: 29,985
Audio matrix: (29985, 9)
Genre matrix: (29985, 4211)
PARQUET + PARTIAL_FIT INCREMENTAL TEST: PASS


## 8. Tổng kết

Sau khi chạy thành công các cell chính, thư mục `artifacts/` cần có model artifact và metadata tương ứng cho lần training hiện tại.

Các kiểm tra hoàn tất trong notebook:
- Dataset validation
- CSV model training
- Artifact save/load thuộc đúng lần chạy hiện tại
- Metadata khớp artifact và không chứa đường dẫn môi trường tuyệt đối
- Track ID ordering
- Recommendation smoke test
- Incremental batch ingestion bằng Parquet
- `StandardScaler.partial_fit()` qua nhiều batch
- Finalize audio/genre matrices

Các phép đánh giá định lượng và biểu đồ được thực hiện riêng trong `notebooks/04_evaluate_model.ipynb` và `src/evaluate.py`.

**Quan trọng:** Sau khi mở notebook, hãy chọn Runtime → Restart session rồi chạy từ đầu đến cuối. Không chạy cell lẻ trên runtime còn giữ module/artifact cũ.
